# Experiment 09: DBSCAN Density-Based Clustering and Noise Detection


### 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score

print("Libraries imported successfully.")


### 2. Generate and Inspect Two-Moons Dataset


In [ ]:
X, true_labels = make_moons(n_samples=500, noise=0.06, random_state=42)

data = pd.DataFrame(X, columns=["Feature 1", "Feature 2"])
print("Dataset shape (samples, features):", data.shape)
display(data.head())

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], s=25)
plt.title("Two-Moons Dataset Before Clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


### 3. Standardize Features


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature means after scaling:", np.round(X_scaled.mean(axis=0), 3))
print("Feature standard deviations:", np.round(X_scaled.std(axis=0), 3))


### 4. Apply DBSCAN (eps=0.25, min_samples=5)


In [ ]:
eps_value = 0.25
min_samples_value = 5

dbscan = DBSCAN(eps=eps_value, min_samples=min_samples_value)
cluster_labels = dbscan.fit_predict(X_scaled)

data["Cluster"] = cluster_labels

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = int(np.sum(cluster_labels == -1))
n_core = len(dbscan.core_sample_indices_)

print("DBSCAN parameters: eps =", eps_value, ", min_samples =", min_samples_value)
print("Estimated clusters:", n_clusters)
print("Core samples      :", n_core)
print("Noise points (-1) :", n_noise)
print(f"Noise percentage  : {100 * n_noise / len(cluster_labels):.2f}%")

display(data["Cluster"].value_counts().sort_index().rename("Number of points").to_frame())


### 5. Visualize Clusters and Noise Points


In [ ]:
plt.figure(figsize=(8, 6))

unique_labels = sorted(set(cluster_labels))
colors = plt.cm.tab10(np.linspace(0, 1, max(len(unique_labels), 1)))

for color, label in zip(colors, unique_labels):
    mask = cluster_labels == label
    if label == -1:
        plt.scatter(
            X[mask, 0], X[mask, 1],
            c="black", marker="x", s=65, label="Noise (-1)"
        )
    else:
        plt.scatter(
            X[mask, 0], X[mask, 1],
            color=color, s=28, label=f"Cluster {label}"
        )

# Outline core samples
core_mask = np.zeros(len(X), dtype=bool)
core_mask[dbscan.core_sample_indices_] = True
plt.scatter(
    X[core_mask, 0], X[core_mask, 1],
    facecolors="none", edgecolors="gray", s=80,
    linewidths=0.7, label="Core samples (outline)"
)

plt.title(f"DBSCAN Clustering (eps={eps_value}, min_samples={min_samples_value})")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


### 6. Investigate Effect of DBSCAN Parameters


In [ ]:
parameter_rows = []

for eps in [0.15, 0.20, 0.25, 0.30, 0.35]:
    model = DBSCAN(eps=eps, min_samples=min_samples_value)
    labels = model.fit_predict(X_scaled)
    cluster_count = len(set(labels)) - (1 if -1 in labels else 0)
    noise_count = int(np.sum(labels == -1))
    
    non_noise = labels != -1
    valid_labels = labels[non_noise]
    valid_X = X_scaled[non_noise]
    if len(np.unique(valid_labels)) >= 2 and len(np.unique(valid_labels)) < len(valid_labels):
        sil = silhouette_score(valid_X, valid_labels)
    else:
        sil = np.nan
        
    parameter_rows.append({
        "eps": eps,
        "min_samples": min_samples_value,
        "Clusters (excluding noise)": cluster_count,
        "Noise points": noise_count,
        "Noise percentage": round(100 * noise_count / len(labels), 1),
        "Silhouette (non-noise only)": sil
    })

parameter_results = pd.DataFrame(parameter_rows)
display(parameter_results.round(3))


### 7. Optional External Comparison: Adjusted Rand Index (ARI)


In [ ]:
ari = adjusted_rand_score(true_labels, cluster_labels)
print(f"Adjusted Rand Index: {ari:.4f}")
